Univariate Analysis

In [0]:
# Import required libraries
from pyspark.sql.functions import col
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
spark.sql("SHOW TABLES IN data_processed.credit_risk_main").show()

In [0]:
# Load Data - Option 1
df = spark.table("data_processed.credit_risk_main.loans_transformed")

In [0]:

%sql
USE CATALOG data_processed;
USE SCHEMA credit_risk_main;

In [0]:
# Load Data (Option 2)
df = spark.table('loans_transformed')



In [0]:
df.show()

In [0]:
# Convert PySpark DataFrame to Pandas for plotting
# (for large datasets, sample first to avoid memory issues)
pdf = df.select(
    "loan_amnt", "funded_amnt", "installment", "term", "grade"
).sample(fraction=0.1, seed=42).toPandas()

Loan Distribution


In [0]:
# ------------------------
# Histograms: loan_amnt, funded_amnt, installment
# ------------------------
plt.figure(figsize=(18, 5))

for i, col_name in enumerate(["loan_amnt", "funded_amnt", "installment"], 1):
    plt.subplot(1, 3, i)
    sns.histplot(pdf[col_name], bins=50, kde=True)
    plt.title(f"Distribution of {col_name}")

plt.tight_layout()
plt.show()

In [0]:
# ------------------------
# Boxplots to detect outliers
# ------------------------
plt.figure(figsize=(18, 5))

for i, col_name in enumerate(["loan_amnt", "funded_amnt", "installment"], 1):
    plt.subplot(1, 3, i)
    sns.boxplot(y=pdf[col_name])
    plt.title(f"Boxplot of {col_name}")

plt.tight_layout()
plt.show()


In [0]:
# ------------------------
# Distribution by term (36 vs 60 months)
# ------------------------
plt.figure(figsize=(10, 5))
sns.countplot(data=pdf, x="term")
plt.title("Distribution of loans by term")
plt.show()


In [0]:
# ------------------------
# Distribution by grade
# ------------------------
plt.figure(figsize=(12, 5))
sns.countplot(data=pdf, x="grade", order=sorted(pdf["grade"].unique()))
plt.title("Distribution of loans by grade")
plt.show()

Notes:

Sampling: For large Spark DataFrames, use .sample() before converting to Pandas. Otherwise, toPandas() may crash.

Histograms: Shows distribution of numeric features. kde=True overlays a density curve.

Boxplots: Helps detect outliers visually.

Countplots: Good for categorical variables like term and grade.

Extending univariate analysis by using violin plots and facet plots with Seaborn to compare distributions of loan_amnt by term and grade together.

In [0]:
# Convert a sample to Pandas (to avoid memory issues)
pdf = df.select("loan_amnt", "term", "grade").sample(fraction=0.1, seed=42).toPandas()

In [0]:
# ------------------------
# Violin plot: loan_amnt by term
# ------------------------
plt.figure(figsize=(8, 6))
sns.violinplot(x="term", y="loan_amnt", data=pdf)
plt.title("Loan Amount Distribution by Term")
plt.ylabel("Loan Amount")
plt.xlabel("Term")
plt.show()

In [0]:
# ------------------------
# Violin plot: loan_amnt by grade
# ------------------------
plt.figure(figsize=(10, 6))
sns.violinplot(x="grade", y="loan_amnt", data=pdf, order=sorted(pdf["grade"].unique()))
plt.title("Loan Amount Distribution by Grade")
plt.ylabel("Loan Amount")
plt.xlabel("Grade")
plt.show()

In [0]:
# ------------------------
# Facet grid: loan_amnt by grade, separated by term
# ------------------------
g = sns.FacetGrid(pdf, col="term", col_order=sorted(pdf["term"].unique()), height=5, aspect=1)
g.map_dataframe(sns.violinplot, x="grade", y="loan_amnt", order=sorted(pdf["grade"].unique()))
g.set_axis_labels("Grade", "Loan Amount")
g.set_titles(col_template="Term: {col_name}")
plt.show()

Violin plots show the distribution and density of loan_amnt for each term or grade.

FacetGrid separates the data by term, then shows violin plots for loan_amnt across grade within each term.

sample(fraction=0.1) ensures large datasets don’t crash when converting to Pandas.

Borrower Profile:

In [0]:
# Import libraries
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col

In [0]:
# Convert selected columns to Pandas (sample if dataset is large)
pdf = df.select("home_ownership", "verification_status", "annual_inc", "addr_state") \
        .sample(fraction=0.1, seed=42).toPandas()

In [0]:
# ------------------------
# Bar plot: home_ownership
# ------------------------
plt.figure(figsize=(6, 5))
sns.countplot(data=pdf, x="home_ownership", order=pdf['home_ownership'].value_counts().index)
plt.title("Distribution of Home Ownership")
plt.ylabel("Number of Loans")
plt.xlabel("Home Ownership")
plt.show()

In [0]:
# ------------------------
# Bar plot: verification_status
# ------------------------
plt.figure(figsize=(6, 5))
sns.countplot(data=pdf, x="verification_status", order=pdf['verification_status'].value_counts().index)
plt.title("Distribution of Verification Status")
plt.ylabel("Number of Loans")
plt.xlabel("Verification Status")
plt.show()

In [0]:
# ------------------------
# Histogram: annual_inc
# ------------------------
plt.figure(figsize=(8, 5))
sns.histplot(pdf['annual_inc'], bins=20, kde=True)
plt.title("Distribution of Annual Income")
plt.xlabel("Annual Income")
plt.ylabel("Number of Borrowers")
plt.show()

In [0]:
# ------------------------
# Top 10 states with most loans (addr_state)
# ------------------------
top_states = pdf['addr_state'].value_counts().nlargest(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_states.index, y=top_states.values)
plt.title("Top 10 States with Most Loans")
plt.xlabel("State")
plt.ylabel("Number of Loans")
plt.show()

In [0]:
# ------------------------
# Boxplot: Annual Income by Home Ownership
# ------------------------
plt.figure(figsize=(8, 5))
sns.boxplot(data=pdf, x="home_ownership", y="annual_inc",
            order=pdf['home_ownership'].value_counts().index)
plt.title("Annual Income by Home Ownership")
plt.ylabel("Annual Income")
plt.xlabel("Home Ownership")
plt.yscale('log')  # log scale for better visualization if there are large outliers
plt.show()

In [0]:
# ------------------------
# Boxplot: Annual Income by Verification Status
# ------------------------
plt.figure(figsize=(8, 5))
sns.boxplot(data=pdf, x="verification_status", y="annual_inc",
            order=pdf['verification_status'].value_counts().index)
plt.title("Annual Income by Verification Status")
plt.ylabel("Annual Income")
plt.xlabel("Verification Status")
plt.yscale('log')
plt.show()


In [0]:
# ------------------------
# State-wise Loan Distribution Percentage
# ------------------------
state_percent = pdf['addr_state'].value_counts(normalize=True).nlargest(10) * 100
plt.figure(figsize=(10, 5))
sns.barplot(x=state_percent.index, y=state_percent.values)
plt.title("Top 10 States Loan Distribution (%)")
plt.xlabel("State")
plt.ylabel("Percentage of Loans")
plt.show()

Features included in this pipeline:

Categorical distributions: home_ownership and verification_status.

Income analysis: histogram and boxplots to detect outliers and compare groups.

Log scale for income boxplots: handles large outliers gracefully.

Top 10 states: number of loans and percentage distribution for deeper insights.

Sampled Pandas DataFrame: prevents memory issues for large Spark DataFrames.

Extending Borrower Profile analysis with facet plots and violin plots to compare annual_inc distributions across multiple dimensions: home_ownership, verification_status, and grade.

Credit Behavior:

In [0]:
# ------------------------
# Define columns to analyze
# ------------------------
numeric_cols = ["revol_bal", "total_acc", "delinq_2yrs"]
int_rate_col = "int_rate"
grade_col = "grade"

In [0]:
# ------------------------
# Filter only existing columns
# ------------------------
existing_cols = [c for c in numeric_cols + [int_rate_col, grade_col] if c in df.columns]
print(f"Columns found in dataset: {existing_cols}")

In [0]:
# ------------------------
# Convert to Pandas (sample for performance)
# ------------------------
pdf = df.select(*existing_cols).sample(fraction=0.1, seed=42).toPandas()

In [0]:
# ------------------------
# Histograms for numeric_cols
# ------------------------
plt.figure(figsize=(18, 5))
for i, col_name in enumerate(numeric_cols, 1):
    if col_name in pdf.columns:
        plt.subplot(1, len(numeric_cols), i)
        sns.histplot(pdf[col_name], bins=50, kde=True)
        plt.title(f"Distribution of {col_name}")
        plt.xlabel(col_name)
        plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [0]:
# ------------------------
# Boxplots for numeric_cols
# ------------------------
plt.figure(figsize=(18, 5))
for i, col_name in enumerate(numeric_cols, 1):
    if col_name in pdf.columns:
        plt.subplot(1, len(numeric_cols), i)
        sns.boxplot(y=pdf[col_name])
        plt.title(f"Boxplot of {col_name}")
plt.tight_layout()
plt.show()


In [0]:
# ------------------------
# Distribution of int_rate by grade
# ------------------------
if int_rate_col in pdf.columns and grade_col in pdf.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=pdf, x=grade_col, y=int_rate_col, order=sorted(pdf[grade_col].unique()))
    plt.title("Interest Rate Distribution by Grade")
    plt.xlabel("Grade")
    plt.ylabel("Interest Rate (%)")
    plt.show()

    plt.figure(figsize=(10, 6))
    sns.violinplot(data=pdf, x=grade_col, y=int_rate_col, order=sorted(pdf[grade_col].unique()))
    plt.title("Interest Rate Distribution by Grade (Violin Plot)")
    plt.xlabel("Grade")
    plt.ylabel("Interest Rate (%)")
    plt.show()

Repayment Info:

In [0]:
# ------------------------
# Define columns for Repayment Info
# ------------------------
repayment_cols = ["total_pymnt", "recoveries", "default_ind"]

In [0]:
# Filter only existing columns
existing_cols = [c for c in repayment_cols if c in df.columns]
print(f"Columns found in dataset: {existing_cols}")

In [0]:
# ------------------------
# Convert to Pandas (sample for performance)
# ------------------------
pdf = df.select(*existing_cols).sample(fraction=0.1, seed=42).toPandas()

In [0]:
# ------------------------
# Histogram: total_pymnt
# ------------------------
if "total_pymnt" in pdf.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(pdf["total_pymnt"], bins=50, kde=True)
    plt.title("Distribution of Total Payment")
    plt.xlabel("Total Payment")
    plt.ylabel("Count")
    plt.show()


In [0]:
# ------------------------
# Histogram: recoveries
# ------------------------
if "recoveries" in pdf.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(pdf["recoveries"], bins=10, kde=True)
    plt.title("Distribution of Recoveries")
    plt.xlabel("Recoveries")
    plt.ylabel("Count")
    plt.show()

In [0]:
# ------------------------
# Frequency of defaults
# ------------------------

# Convert to Pandas (sample for performance)
if "default_ind" in df.columns:
    pdf = df.select("default_ind").sample(fraction=0.1, seed=42).toPandas()

    # Count plot (bar chart)
    plt.figure(figsize=(6, 5))
    sns.countplot(data=pdf, x="default_ind", order=pdf['default_ind'].value_counts().index)
    plt.title("Frequency of Defaults")
    plt.xlabel("Default Indicator")
    plt.ylabel("Number of Loans")
    plt.show()

    # Optional: percentage distribution
    default_percent = pdf['default_ind'].value_counts(normalize=True) * 100
    plt.figure(figsize=(6, 5))
    sns.barplot(x=default_percent.index, y=default_percent.values)
    plt.title("Percentage of Defaults")
    plt.xlabel("Default Indicator")
    plt.ylabel("Percentage (%)")
    plt.show()